In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
data = df['Price'].values.reshape(-1,1)

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)
def create_sequences(data, time_step=60):
    X, y = [], []
    for i in range(time_step, len(data)):
        X.append(data[i-time_step:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

time_step = 60
X, y = create_sequences(data_scaled, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)  

train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

model = Sequential()
model.add(Input(shape=(time_step, 1)))
model.add(LSTM(50, return_sequences=True))
model.add(LSTM(50))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, y_train, epochs=25, batch_size=32, verbose=1)
y_pred = model.predict(X_test)

y_pred_inv = scaler.inverse_transform(y_pred)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1,1))
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))

mae = mean_absolute_error(y_test_inv, y_pred_inv)

r2 = r2_score(y_test_inv, y_pred_inv)

mape = np.mean(np.abs((y_test_inv - y_pred_inv) / y_test_inv))

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.4f}")
print("MAPE:", mape)
#Manual MAPE
epoch_list = [10, 20, 25]
unit_list = [50, 100]

results = []

for epochs in epoch_list:
    for units in unit_list:

        model = Sequential()
        model.add(Input(shape=(time_step,1)))
model.add(LSTM(units, return_sequences=True))
model.add(LSTM(units))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, y_train, epochs=epochs, batch_size=32, verbose=0)

y_pred = model.predict(X_test)
y_pred_inv = scaler.inverse_transform(y_pred)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1,1))
        
mape = np.mean(np.abs((y_test_inv - y_pred_inv) / y_test_inv))

results.append((epochs, units, mape))

print(f"Epochs: {epochs}, Units: {units}, MAPE: {mape:.6f}")


Epoch 1/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 9.6919e-04
Epoch 2/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 5.1602e-05
Epoch 3/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 4.9032e-05
Epoch 4/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 4.6909e-05
Epoch 5/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 4.3527e-05
Epoch 6/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 4.2782e-05
Epoch 7/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 4.6811e-05
Epoch 8/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 4.0349e-05
Epoch 9/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 4.4438e-05
Epoch 10/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.5856e-05
Epoch 11/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.6741e-05
Epoch 12/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 3.5944e-05
Epoch 13/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.2267e-05
Epoch 14/25
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.3037e-05
Epoch 15/25
77/

In [18]:
os.makedirs("models", exist_ok=True)
model.save("models/lstm_model.keras")